# 🔬 Weakly-Supervised Particle Picking in Cryo-EM Micrographs

---

## Overview

**Particle picking** is the first computational step of single-particle
cryo-EM: locating every copy of the target protein in a raw micrograph
so it can later be extracted, aligned, and averaged into a 3D structure.
Micrographs have extremely low signal-to-noise ratio, and — critically —
**nobody has a ground-truth particle map**: even expert manual picking is
just one more noisy label source. This makes particle picking a genuine
**weakly-supervised / positive-unlabeled learning** problem, which is
exactly why modern tools (Topaz, crYOLO, DeepPicker) had to depart from
ordinary supervised classification.

This notebook builds that pipeline end-to-end on **real published cryo-EM
micrographs**:

| Module | Topic |
|--------|-------|
| **1**  | Real Micrographs & Classical Template/Blob Picking (the pre-deep-learning baseline) |
| **2**  | Why Particle Picking Is Positive-Unlabeled Learning |
| **3**  | Patch-CNN Classifier Trained on Bootstrap Labels |
| **4**  | Full-Micrograph Sliding-Window Inference & Peak Detection |
| **5**  | Production Methods & Limitations |

### Dataset

We use real **apoferritin** cryo-EM micrographs derived from **EMPIAR-10146**
(the cisTEM apoferritin tutorial dataset), mirrored in the
`jianlin-cheng/DeepCryoEM` GitHub repository — 20 real 16-bit micrographs,
each ~1200×1240 px, motion-corrected frame averages. Apoferritin is a
small, roughly spherical, highly symmetric protein — a classic picking
benchmark because its near-circular projection is easy to reason about
visually while still exhibiting realistic low-contrast cryo-EM noise.

> **Prerequisites:** `numpy`, `scipy`, `torch`, `scikit-image`, `Pillow`.
> All cells are self-contained; the dataset auto-downloads on first run.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# torch imported at top-level to avoid NameError if cells run out of order.
# ============================================================
import os
import glob
import warnings
import urllib.request
from urllib.parse import quote

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import torch
import torch.nn as nn
from skimage.feature import blob_log, peak_local_max
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")
np.random.seed(0)
torch.manual_seed(0)

DEVICE = "cpu"  # sliding-window inference here is fast enough on CPU
print("Device:", DEVICE)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — DOWNLOAD REAL CRYO-EM MICROGRAPHS (EMPIAR-10146 apoferritin)
# ============================================================
DATA_DIR = "apo_data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = ("https://raw.githubusercontent.com/jianlin-cheng/DeepCryoEM/"
            "master/APFIRITIN%20DATASET/")
FILENAMES = [
    "May08_03.05.02.bin_avg.png", "May08_03.13.02.bin_avg.png", "May08_03.15.02.bin_avg.png",
    "May08_03.43.44.bin_avg.png", "May08_04.32.27.bin_avg.png", "May08_04.33.57.bin_avg.png",
    "May08_04.36.57.bin_avg.png", "May08_05.24.39.bin_avg.png", "May08_05.31.00.bin_avg.png",
    "May08_05.35.20.bin_avg.png", "May08_05.53.51.bin_avg.png", "May08_05.55.31.bin_avg.png",
    "May08_05.57.01.bin_avg.png", "May08_05.59.52.bin_avg.png", "May08_06.09.02.bin_avg.png",
    "May08_06.30.43.bin_avg.png", "May08_07.08.55.bin_avg.png", "May08_07.10.35.bin_avg.png",
    "May08_07.13.35.bin_avg.png", "May08_07.30.46.bin_avg.png",
]
for fname in FILENAMES:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(BASE_URL + quote(fname), fpath)
        except Exception as e:
            print(f"  Warning: could not fetch {fname}: {e}")


def load_normalized(fpath):
    """Load a 16-bit micrograph and percentile-normalize to [0,1]."""
    arr = np.array(Image.open(fpath)).astype(np.float32)
    p1, p99 = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - p1) / (p99 - p1), 0, 1)


slice_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.png")))
micrographs = [load_normalized(f) for f in slice_files]
print(f"Loaded {len(micrographs)} real apoferritin micrographs "
      f"(EMPIAR-10146, cisTEM tutorial dataset)")
print("Micrograph shape:", micrographs[0].shape)

train_micrographs = micrographs[:16]
eval_micrographs = micrographs[16:]

## 1.1 The Classical Baseline: Template / Blob-Based Picking

Before deep learning, particle picking relied on **template matching**
(normalized cross-correlation against a reference projection) or
**blob detectors** exploiting the fact that small globular particles
under phase-contrast defocus appear as approximately circular
dark blobs against a lighter vitreous-ice background. We use a
**Laplacian-of-Gaussian (LoG) blob detector** — a direct descendant of
the matched-filter idea used in tools like XMIPP and the APPLE picker —
as our classical baseline and label bootstrap source.

$$\text{LoG}(x,y;\sigma) = \sigma^2\nabla^2\bigl(G_\sigma * I\bigr)(x,y)$$

Local maxima of the (scale-normalized) LoG response across a range of
$\sigma$ correspond to blob-like structures whose size matches
$\sigma$ — exactly the apoferritin particle radius in pixels here.

In [ ]:
# ============================================================
# MODULE 1 — CLASSICAL LoG BLOB PICKER
# ============================================================
def classical_pick(img, min_sigma=4, max_sigma=10, threshold=0.1):
    """LoG blob detector on the inverted image (apoferritin appears dark)."""
    inverted = 1 - img
    blobs = blob_log(inverted, min_sigma=min_sigma, max_sigma=max_sigma,
                      num_sigma=5, threshold=threshold, overlap=0.2)
    return blobs[:, :2]  # (y, x) centers


demo_img = train_micrographs[0]
demo_picks = classical_pick(demo_img)
print(f"Classical picker found {len(demo_picks)} candidate particles "
      f"in the demo micrograph")

# ------------------------------------------------------------------
# VISUALIZATION 1 — Classical picks overlaid on a real micrograph
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle("Module 1 — Classical LoG Blob Picking on a Real Cryo-EM Micrograph\n"
             "(EMPIAR-10146 apoferritin, cisTEM tutorial dataset)",
             color=ACCENT, fontweight="bold")
axes[0].imshow(demo_img, cmap="gray")
axes[0].set_title("Raw micrograph", color=TEXT, fontsize=10)
axes[1].imshow(demo_img, cmap="gray")
for y, x in demo_picks:
    axes[1].add_patch(plt.Circle((x, y), 10, color="#f0883e", fill=False, lw=1))
axes[1].set_title(f"Classical picks ({len(demo_picks)} particles)", color=TEXT, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 2 — Why Particle Picking Is Positive-Unlabeled Learning

## 2.1 The Missing-Negatives Problem

Suppose we treat every classical pick as a "particle" label and every
other patch as "background." This is **not** a clean binary
classification problem: the classical picker (and even human experts)
**miss real particles** — overlapping particles, low-contrast particles
near the defocus zero-crossing, or particles at the image edge. Any
"background" patch sampled near a missed particle is a **false
negative** in our training set, not a true one.

## 2.2 How Real Tools Handle This

- **Topaz** (Bepler et al. 2019) explicitly models this as
  positive-unlabeled (PU) learning: it assumes only a fraction $\pi$ of
  "unlabeled" patches are truly negative and corrects the loss
  accordingly (a GE-binomial / PU-learning correction), rather than
  trusting every unlabeled patch as a clean negative.
- **crYOLO** (Wagner et al. 2019) sidesteps per-patch classification
  entirely, using a YOLO-style detection head trained end-to-end on
  sparse expert picks, tolerating missed labels via its objectness loss.

For this tutorial we use the **simpler, more transparent classical
supervised route** (train a plain positive/negative patch classifier),
but the noisy-label caveat above is exactly why production tools use
the more careful formulations — we return to this in Module 5.

---
# Module 3 — Patch-CNN Classifier Trained on Bootstrap Labels

## 3.1 Building the Training Set

- **Positive patches**: 32×32 crops centered on classical picks
- **Negative patches**: 32×32 crops from random locations at least
  20 px from any classical pick (a heuristic to reduce — but not
  eliminate — the missing-negatives problem from Module 2)

## 3.2 Loss

Standard binary cross-entropy over patches:
$$\mathcal{L} = -\frac{1}{N}\sum_i \bigl[y_i\log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\bigr]$$

In [ ]:
# ============================================================
# MODULE 3 — BUILD PATCH DATASET FROM CLASSICAL BOOTSTRAP LABELS
# ============================================================
PATCH = 32


def extract_patch(img, y, x, size=PATCH):
    h, w = img.shape
    y, x = int(y), int(x)
    half = size // 2
    if y - half < 0 or x - half < 0 or y + half >= h or x + half >= w:
        return None
    return img[y - half:y + half, x - half:x + half]


rng = np.random.RandomState(1)
pos_patches, neg_patches = [], []
all_train_picks = [classical_pick(im) for im in train_micrographs]

for img, coords in zip(train_micrographs, all_train_picks):
    for (y, x) in coords:
        p = extract_patch(img, y, x)
        if p is not None:
            pos_patches.append(p)
    got, tries = 0, 0
    n_neg_target = len(coords)
    while got < n_neg_target and tries < n_neg_target * 10:
        tries += 1
        y = rng.randint(PATCH, img.shape[0] - PATCH)
        x = rng.randint(PATCH, img.shape[1] - PATCH)
        if len(coords) > 0:
            dist = np.sqrt(((coords - np.array([y, x])) ** 2).sum(1)).min()
            if dist < 20:
                continue
        p = extract_patch(img, y, x)
        if p is not None:
            neg_patches.append(p)
            got += 1

print(f"Bootstrap training set: {len(pos_patches)} positive, "
      f"{len(neg_patches)} negative patches from {len(train_micrographs)} micrographs")

X = np.stack(pos_patches + neg_patches)[:, None, :, :].astype(np.float32)
y_lab = np.array([1] * len(pos_patches) + [0] * len(neg_patches), dtype=np.float32)
perm = rng.permutation(len(X))
X, y_lab = X[perm], y_lab[perm]

In [ ]:
# ============================================================
# MODULE 3 — PATCH-CNN CLASSIFIER & TRAINING LOOP
# ============================================================
class PatchCNN(nn.Module):
    """Small CNN classifying 32x32 patches as particle / background."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, 1)

    def forward(self, x):
        f = self.features(x).flatten(1)
        return self.classifier(f)


model = PatchCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

Xt = torch.from_numpy(X).to(DEVICE)
yt = torch.from_numpy(y_lab).to(DEVICE)
n_samples = len(Xt)
batch_size = 32
n_epochs = 8

epoch_losses = []
for epoch in range(n_epochs):
    idx = torch.randperm(n_samples)
    running = 0.0
    n_batches = 0
    for i in range(0, n_samples, batch_size):
        b = idx[i:i + batch_size]
        optimizer.zero_grad()
        out = model(Xt[b]).squeeze(1)
        loss = loss_fn(out, yt[b])
        loss.backward()
        optimizer.step()
        running += loss.item()
        n_batches += 1
    epoch_losses.append(running / n_batches)
    print(f"epoch {epoch+1}/{n_epochs} | BCE loss = {epoch_losses[-1]:.4f}")

# ------------------------------------------------------------------
# VISUALIZATION 2 — Training loss curve
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epoch_losses, marker="o", color=ACCENT, lw=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE loss")
ax.set_title("Module 3 — Patch-CNN Training Loss (trained on classical bootstrap labels)",
             color=ACCENT, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Module 4 — Full-Micrograph Sliding-Window Inference

We slide the trained patch classifier over an entire **held-out**
micrograph (never used for bootstrap labels or training) to build a
dense particle-probability heatmap, then apply local-maximum
suppression to get final particle coordinates — the same two-stage
design (dense scoring → peak detection) used by real CNN-based pickers
before the YOLO-style single-shot detectors like crYOLO.

## Evaluation Caveat

Just as in Module 2, our classical picks are **not perfect ground
truth** — so the comparison below measures **agreement between two
imperfect pickers**, not accuracy against a true reference. This
mirrors how new picking methods are actually validated in practice:
against existing picks or downstream reconstruction quality, since no
perfect particle map exists for a real micrograph.

In [ ]:
# ============================================================
# MODULE 4 — SLIDING-WINDOW HEATMAP + PEAK DETECTION
# ============================================================
def sliding_window_heatmap(model, img, patch=PATCH, stride=4, batch_size=512):
    h, w = img.shape
    half = patch // 2
    heat = np.zeros((h, w), dtype=np.float32)
    ys = list(range(half, h - half, stride))
    xs = list(range(half, w - half, stride))
    coords, patches = [], []
    for y in ys:
        for x in xs:
            patches.append(img[y - half:y + half, x - half:x + half])
            coords.append((y, x))
    patches = np.stack(patches)[:, None, :, :].astype(np.float32)
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(patches), batch_size):
            batch_t = torch.from_numpy(patches[i:i + batch_size]).to(DEVICE)
            preds.append(torch.sigmoid(model(batch_t)).squeeze(1).cpu().numpy())
    preds = np.concatenate(preds)
    for (y, x), p in zip(coords, preds):
        heat[y, x] = p
    return heat


eval_img = eval_micrographs[0]
heatmap = sliding_window_heatmap(model, eval_img, stride=4)
cnn_picks = peak_local_max(heatmap, min_distance=8, threshold_abs=0.5)
classical_eval_picks = classical_pick(eval_img)

print(f"Held-out micrograph — classical picks: {len(classical_eval_picks)}   "
      f"CNN picks: {len(cnn_picks)}")

if len(cnn_picks) > 0 and len(classical_eval_picks) > 0:
    tree = cKDTree(cnn_picks)
    dist_to_cnn, _ = tree.query(classical_eval_picks[:, :2])
    recall_like = (dist_to_cnn < 8).mean()

    tree2 = cKDTree(classical_eval_picks[:, :2])
    dist_to_classical, _ = tree2.query(cnn_picks)
    precision_like = (dist_to_classical < 8).mean()

    print(f"Agreement (classical picks recovered by CNN, within 8px): {recall_like*100:.1f}%")
    print(f"Agreement (CNN picks matching a classical pick, within 8px): {precision_like*100:.1f}%")

# ------------------------------------------------------------------
# VISUALIZATION 3 — Heatmap + pick comparison on held-out micrograph
# ------------------------------------------------------------------
fig = plt.figure(figsize=(18, 6))
fig.suptitle("Module 4 — CNN vs. Classical Picking on a Held-Out Real Micrograph",
             color=ACCENT, fontweight="bold")
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.25)

ax0 = fig.add_subplot(gs[0])
ax0.imshow(eval_img, cmap="gray")
for y, x in classical_eval_picks:
    ax0.add_patch(plt.Circle((x, y), 10, color="#f0883e", fill=False, lw=1))
ax0.set_title(f"Classical LoG picks ({len(classical_eval_picks)})", color=TEXT, fontsize=10)
ax0.axis("off")

ax1 = fig.add_subplot(gs[1])
ax1.imshow(heatmap, cmap="inferno")
ax1.set_title("CNN particle-probability heatmap", color=TEXT, fontsize=10)
ax1.axis("off")

ax2 = fig.add_subplot(gs[2])
ax2.imshow(eval_img, cmap="gray")
for y, x in cnn_picks:
    ax2.add_patch(plt.Circle((x, y), 10, color=ACCENT, fill=False, lw=1))
ax2.set_title(f"CNN picks after peak detection ({len(cnn_picks)})", color=TEXT, fontsize=10)
ax2.axis("off")
plt.tight_layout()
plt.show()

---
# Module 5 — Production Methods & Limitations

| Method | Learning Setup | Notes |
|--------|-----------------|-------|
| Template matching / XMIPP | Cross-correlation, no learning | Classical baseline used here for bootstrap labels |
| APPLE Picker (Heimowitz 2018) | Statistical outlier detection | No manual picks needed, fully classical |
| DeepPicker (Wang 2016) | Supervised CNN classifier | Early deep-learning picker, assumes clean labels |
| Topaz (Bepler 2019) | Positive-unlabeled CNN | Explicitly corrects for missing-negative bias (Module 2) |
| crYOLO (Wagner 2019) | YOLO-style single-shot detector | End-to-end box regression, no sliding window needed |

## Known Limitations of This Tutorial
- Bootstrap "positive" labels come from an imperfect classical picker,
  not expert-verified ground truth — errors and biases in that picker
  propagate directly into the CNN's training signal.
- Negative patches are sampled as "not near a classical pick," which
  only partially addresses the missing-negatives problem from Module 2;
  real PU-learning corrections (Topaz's GE-binomial loss) are not
  implemented here for simplicity.
- Sliding-window inference (Module 4) is computationally wasteful
  compared to detection-head architectures (crYOLO); it is used here
  for pedagogical transparency.
- Apoferritin's near-spherical, high-symmetry shape makes this an
  easier picking target than elongated or highly heterogeneous
  particles.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🔬 Weakly-Supervised Cryo-EM Particle Picking — Pipeline Dashboard",
             fontsize=16, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(demo_img, cmap="gray"); ax0.set_title("Real micrograph (source)", color=TEXT, fontsize=10); ax0.axis("off")
ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(demo_img, cmap="gray")
for y, x in demo_picks:
    ax1.add_patch(plt.Circle((x, y), 10, color="#f0883e", fill=False, lw=1))
ax1.set_title("Classical LoG picks (bootstrap labels)", color=TEXT, fontsize=10); ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 2]); ax2.imshow(heatmap, cmap="inferno"); ax2.set_title("CNN probability heatmap (held-out)", color=TEXT, fontsize=10); ax2.axis("off")
ax3 = fig.add_subplot(gs[0, 3]); ax3.imshow(eval_img, cmap="gray")
for y, x in cnn_picks:
    ax3.add_patch(plt.Circle((x, y), 10, color=ACCENT, fill=False, lw=1))
ax3.set_title("Final CNN picks (held-out)", color=TEXT, fontsize=10); ax3.axis("off")

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.plot(epoch_losses, marker="o", color=ACCENT, lw=2)
ax4.set_title("Training loss (BCE on bootstrap labels)", color=TEXT, fontsize=10)
ax4.set_xlabel("epoch")

ax5 = fig.add_subplot(gs[1, 2])
ax5.bar(["Classical", "CNN"], [len(classical_eval_picks), len(cnn_picks)], color=["#f0883e", ACCENT])
ax5.set_title("Pick count (held-out micrograph)", color=TEXT, fontsize=10)

ax6 = fig.add_subplot(gs[1, 3])
if len(cnn_picks) > 0 and len(classical_eval_picks) > 0:
    ax6.bar(["Classical recovered\nby CNN", "CNN matching\na classical pick"],
            [recall_like * 100, precision_like * 100], color=["#3fb950", "#a371f7"])
    ax6.set_ylabel("% within 8px")
ax6.set_title("Agreement between pickers", color=TEXT, fontsize=10)

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Real data | 1 | Real EMPIAR-10146 apoferritin micrographs, not synthetic phantoms |
| Classical baseline | 1 | LoG blob detection — the matched-filter predecessor to deep-learning pickers |
| Problem framing | 2 | Particle picking is positive-unlabeled learning; missed picks corrupt naive negative sampling |
| Model & training | 3 | Small patch-CNN trained on bootstrap labels with BCE loss |
| Inference & evaluation | 4 | Sliding-window heatmap + peak detection on held-out micrographs; agreement-based evaluation (no perfect ground truth exists) |
| Context | 5 | Positioned against APPLE picker, DeepPicker, Topaz, crYOLO |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| LoG blob detection | $\mathcal{O}(N^2 \log \sigma_{\max})$ | Scale-space construction |
| Patch-CNN training | $\mathcal{O}(N_{\text{patches}} \cdot P^2 \cdot C)$ | Small; trains in seconds |
| Sliding-window inference | $\mathcal{O}\!\left(\frac{N^2}{s^2}\cdot P^2 \cdot C\right)$ | Dominant; $s$=stride, quadratic in image size |
| Peak detection | $\mathcal{O}(N^2)$ | Negligible |

## Key References
- Bepler et al. (2019) — Topaz: positive-unlabeled CNN particle picking (*Nature Methods*)
- Wagner et al. (2019) — crYOLO: fast, single-shot particle picking (*Communications Biology*)
- Wang et al. (2016) — DeepPicker: supervised CNN particle picking
- Heimowitz, Andén & Singer (2018) — APPLE picker: automated classical picking
- Al-Azzawi et al. (2019/2020) — DeepCryoPicker (source of the EMPIAR-10146 mirror used here)